# HDF5 files of the CG26 correlation model

Writes the CG26 correlation model (reverse faulting; all four tectonic regions and the total,
between-event, site-to-site and within-site-within-event residual correlations) to the two
HDF5 files used to implement it in other software:

- `clemett_corr_coeffs_openquake.hdf5`: the implementation of the correlation model in
  OpenQuake (PGA and SA only, with the periods stored per region);
- `clemett_corr_coeffs_pickagm.hdf5`: the implementation in `pickagm` (all IMs, with the IM
  labels stored as an attribute of each matrix).

Both are built from the positive-definite correlation csvs that nb `002-im_correlations`
writes to `results/02_im_correlation_model/{fault type}/`, and are written to
`results/02_im_correlation_model/`. Coefficients are rounded to 3 decimals.

In [1]:
import json
import h5py
import numpy as np
import pandas as pd

from phd_project.config.config import load_config
from phd_project.scripts.WP1_ground_motion_set.correlation import (
    load_positive_definite_correlations,
    )

cfg = load_config()
RESULTS = cfg["results"]["corr_model"]

# folder the two HDF5 files are written to
HDF5_FOLDER = RESULTS

In [2]:
# Load the positive-definite model for reverse faulting from the csvs written by nb 002
cg26 = load_positive_definite_correlations(RESULTS, "reverse")
cg26["asc"].keys()

dict_keys(['rho_total', 'rho_dBe', 'rho_dS2Ss', 'rho_dWSes', 'rho_total_B'])

In [3]:
def print_hdf5_tree(name, obj):
    """
    Recursive visitor function to print the HDF5 structure.
    """
    shift = name.count('/') * '    '
    item_type = "[Group]" if isinstance(obj, h5py.Group) else "[Dataset]"
    print(f"{shift}{item_type} {name.split('/')[-1]}")

In [4]:
# create hdf5 files for each correlation model for implementation in openquake
rename_mapper = {
    "rho_total": "total",
    "rho_dBe": "dB_e",
    "rho_dS2Ss": "dS2S_s",
    "rho_dWSes": "dWS_es"
}

with h5py.File(HDF5_FOLDER / "clemett_corr_coeffs_openquake.hdf5", "w") as f:
    for k, v in cg26.items():

        idx = pd.IndexSlice
        periods = np.array(cg26[k]["rho_total"].loc[idx[:, "SA", :], idx[:, "SA", :]].index.get_level_values(2))
        periods = np.array([0] + [float(t) for t in periods])
        if k == "vran":
            k = "vrancea"

        corr_coeffs = f.create_group(k)
        corr_coeffs.create_dataset("periods", data=periods)

        for residual_type, rho_data in v.items():
            if residual_type in rename_mapper.keys():
                rho_data = rho_data.loc[idx[:, ["PGA", "SA"], :], idx[:, ["PGA", "SA"], :]].to_numpy()
                rho_data = np.round(rho_data, 3)
                corr_coeffs.create_dataset(rename_mapper[residual_type], data=rho_data) 
    

In [5]:
filename = HDF5_FOLDER / "clemett_corr_coeffs_openquake.hdf5"
with h5py.File(filename, "r") as f:
    print(f"Root: {filename}")
    f.visititems(print_hdf5_tree)

Root: C:\Users\clemettn\Documents\phd\results\02_im_correlation_model\clemett_corr_coeffs_openquake.hdf5
[Group] asc
    [Dataset] dB_e
    [Dataset] dS2S_s
    [Dataset] dWS_es
    [Dataset] periods
    [Dataset] total
[Group] sinter
    [Dataset] dB_e
    [Dataset] dS2S_s
    [Dataset] dWS_es
    [Dataset] periods
    [Dataset] total
[Group] sslab
    [Dataset] dB_e
    [Dataset] dS2S_s
    [Dataset] dWS_es
    [Dataset] periods
    [Dataset] total
[Group] vrancea
    [Dataset] dB_e
    [Dataset] dS2S_s
    [Dataset] dWS_es
    [Dataset] periods
    [Dataset] total


In [6]:
# create the hdf5 files for implementation in pickagm
with h5py.File(HDF5_FOLDER / "clemett_corr_coeffs_pickagm.hdf5", "w") as f:
    for k, v in cg26.items():
        if k == "vran":
            k = "vrancea"      

        corr_coeffs = f.create_group(k)

        for residual_type, rho_data in v.items():
            if residual_type not in rename_mapper.keys():
                continue

            imts = tuple([(l[1], l[2]) for l in rho_data.index])
            json_string = json.dumps(imts)
            rho_data = rho_data.to_numpy()
            rho_data = np.round(rho_data, 3)
            dset =corr_coeffs.create_dataset(rename_mapper[residual_type], data=rho_data) 
            dset.attrs["imts"] = json_string